# Aurisign FIlipino Sign Language Gesture Recognition 1D CNN Numbers Training

## Dependency Installation and Imports

In [1]:
!pip install opencv-python mediapipe==0.10.14 matplotlib seaborn tensorflow scikit-learn tensorflowjs


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# %%
import cv2
import mediapipe as mp
import numpy as np
import os
from time import time

2026-08-22 22:22:33.220045: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-22 22:22:33.324663: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-22 22:22:33.437548: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787408553.525104   59018 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787408553.551362   59018 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787408553.721978   59018 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

### Data Collection

In [3]:
# %%
# Motion gesture data collection (J, Z) — captures short SEQUENCES instead of single frames.
# Press "c" to record one full sequence for the current letter, "q" to move to the next letter.
MOTION_CLASSES = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]   # full set — used everywhere downstream

MOTION_DATA_PATH = "number_dataset"
SEQ_LEN = 24                 # ~0.8-1s of motion at a typical webcam framerate
SAMPLES_PER_GESTURE = 100    # sequences, not frames
CENTER_BY_WRIST = True
INCLUDE_HANDEDNESS = True
NUM_FEATURES = 64 if INCLUDE_HANDEDNESS else 63

os.makedirs(MOTION_DATA_PATH, exist_ok=True)
for g in MOTION_CLASSES:
    os.makedirs(os.path.join(MOTION_DATA_PATH, g), exist_ok=True)

mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)

cap = cv2.VideoCapture(0)

def get_frame_keypoints(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)
    if not result.multi_hand_landmarks:
        return None, frame
    lm = result.multi_hand_landmarks[0]
    mp_draw.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
    pts = np.array([[p.x, p.y, p.z] for p in lm.landmark])
    if CENTER_BY_WRIST:
        pts -= pts[0]
    pts = pts.flatten()
    if INCLUDE_HANDEDNESS and result.multi_handedness:
        hl = result.multi_handedness[0].classification[0].label
        pts = np.append(pts, 0 if hl == "Left" else 1)
    return pts, frame

for gesture in MOTION_CLASSES:
    print(f"Collecting motion: {gesture} — [c]=record one {SEQ_LEN}-frame sequence, [q]=next letter")
    count = 0
    while count < SAMPLES_PER_GESTURE:
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.flip(frame, 1)
        _, frame = get_frame_keypoints(frame.copy())
        cv2.putText(frame, f"{gesture} {count}/{SAMPLES_PER_GESTURE}  [c]=record  [q]=next",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imshow("Collect Motion Gestures", frame)
        k = cv2.waitKey(1) & 0xFF

        if k == ord("c"):
            sequence = []
            print(f"  Recording {gesture} #{count}... perform the gesture now")
            while len(sequence) < SEQ_LEN:
                ret, f2 = cap.read()
                if not ret:
                    continue
                f2 = cv2.flip(f2, 1)
                kp, f2 = get_frame_keypoints(f2)
                # a short mid-gesture occlusion shouldn't corrupt the whole window,
                # so hold the last known keypoints instead of dropping the frame
                if kp is None and sequence:
                    kp = sequence[-1]
                elif kp is None:
                    kp = np.zeros(NUM_FEATURES)
                sequence.append(kp)
                cv2.putText(f2, f"RECORDING {len(sequence)}/{SEQ_LEN}", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                cv2.imshow("Collect Motion Gestures", f2)
                cv2.waitKey(1)
            np.save(os.path.join(MOTION_DATA_PATH, gesture, f"{count:03d}.npy"), np.array(sequence))
            count += 1

        if k == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()
print("Done collecting motion gestures!")

I0000 00:00:1787408561.694023   59018 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1787408561.704794   63315 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 620 (WHL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787408561.735364   63303 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787408561.752152   63302 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


QFontDatabase: Cannot find font directory /home/bennybun29/Documents/jupyter_projects/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/bennybun29/Documents/jupyter_projects/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/bennybun29/Documents/jupyter_projects/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/bennybun29/Documents/jupyter_projects/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ fo

  Recording 0 #0... perform the gesture now
  Recording 0 #1... perform the gesture now
  Recording 0 #2... perform the gesture now
  Recording 0 #3... perform the gesture now
  Recording 0 #4... perform the gesture now
  Recording 0 #5... perform the gesture now
  Recording 0 #6... perform the gesture now
  Recording 0 #7... perform the gesture now
  Recording 0 #8... perform the gesture now
  Recording 0 #9... perform the gesture now
  Recording 0 #10... perform the gesture now
  Recording 0 #11... perform the gesture now
  Recording 0 #12... perform the gesture now
  Recording 0 #13... perform the gesture now
  Recording 0 #14... perform the gesture now
  Recording 0 #15... perform the gesture now
  Recording 0 #16... perform the gesture now
  Recording 0 #17... perform the gesture now
  Recording 0 #18... perform the gesture now
  Recording 0 #19... perform the gesture now
  Recording 0 #20... perform the gesture now
  Recording 0 #21... perform the gesture now
  Recording 0 #22...

### Number Data Loading and Preprocessing

In [4]:
# %%
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

X_motion, y_motion = [], []
for i, gesture in enumerate(MOTION_CLASSES):
    folder = os.path.join(MOTION_DATA_PATH, gesture)
    for f in os.listdir(folder):
        X_motion.append(np.load(os.path.join(folder, f)))
        y_motion.append(i)

X_motion = np.array(X_motion)   # shape: (num_samples, SEQ_LEN, NUM_FEATURES)
y_motion = to_categorical(y_motion, len(MOTION_CLASSES))

Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_motion, y_motion, test_size=0.2, random_state=42)
print("Train:", Xm_train.shape, "Test:", Xm_test.shape)

Train: (880, 24, 64) Test: (220, 24, 64)


### Number Model

In [5]:
# %%
motion_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, X_motion.shape[-1])),
    tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(MOTION_CLASSES), activation='softmax'),
])
motion_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
motion_model.summary()

2026-08-22 22:51:35.374991: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 24, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 24, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,579 (115.54 KB)

 Trainable params: 29,579 (115.54 KB)

 Non-trainable params: 0 (0.00 B)

### Number Model Training

In [6]:
# %%
motion_history = motion_model.fit(Xm_train, ym_train, validation_data=(Xm_test, ym_test),
                                   epochs=50, batch_size=8)

Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2455 - loss: 2.2031 - val_accuracy: 0.4682 - val_loss: 1.8796
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4409 - loss: 1.5084 - val_accuracy: 0.6909 - val_loss: 1.1115
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7091 - loss: 0.8648 - val_accuracy: 0.8182 - val_loss: 0.6565
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7795 - loss: 0.5708 - val_accuracy: 0.8318 - val_loss: 0.4752
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8580 - loss: 0.3890 - val_accuracy: 0.8909 - val_loss: 0.3861
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8591 - loss: 0.3415 - val_accuracy: 0.8545 - val_loss: 0.3563
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9080 - loss: 0.2492 - val_accuracy: 0.9364 - val_loss: 0.2717
Epoch 8/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9432 - loss: 0.1703 - val_accuracy: 0.

### Number Model Export

In [7]:
# %%
motion_model.export("number_model")
print("Number SavedModel export complete!")

# In Google Colab, convert this the same way you convert the static model,
# just pointed at the motion folders so it doesn't overwrite the static tfjs export:
# !tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model number_model tfjs_number_model

INFO:tensorflow:Assets written to: number_model/assets


INFO:tensorflow:Assets written to: number_model/assets


Saved artifact at 'number_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 24, 64), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  132781062077520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132781062078096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132780851053200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132780851054736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132780851053584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132780851054928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132780851054352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132780851055120: TensorSpec(shape=(), dtype=tf.resource, name=None)
Number SavedModel export complete!
